In [3]:
#!/usr/bin/env python3
# =============================================================================
# Minimal main loop (UTC+0 / epoch-ms) with base-table sync
# =============================================================================
# - Periodically reads the DB and prints a neat, tabulated view of:
#     * current UTC time in epoch milliseconds (canonical project unit)
#     * max open_time for base, dev and final (logreg) tables (interpreted as UTC)
#     * max open_time_ms for the base table (if present) and its readable UTC string
# - Additionally: calls sync_bchusdt_1m_from_ms(...) for the base table only.
# - Uses utils._load_config() and the UTC/ms helpers in utils.py
# - All times and conversions are UTC-only (no local time usage).
# - No pred_view or plotting here; intentionally minimal and easy to read.
# =============================================================================

from typing import Optional
import time
import sqlite3

import utils as utils  # contains now_utc_ms(), now_utc_str(), ms_to_utc_str(), utc_str_to_ms()
from database_codes.bch_usdt_1m import sync_bchusdt_1m_from_ms

# Poll interval (seconds)
POLL_SECONDS = 10


# -----------------------------------------------------------------------------
# Helper: safe SQL MAX fetch
# -----------------------------------------------------------------------------
def _sql_max(conn: sqlite3.Connection, table: str, column: str) -> Optional[object]:
    # Return MAX(column) from table or None if table/column missing.
    try:
        cur = conn.cursor()
        cur.execute(f'SELECT MAX("{column}") FROM "{table}"')
        row = cur.fetchone()
        return row[0] if row and row[0] is not None else None
    except sqlite3.OperationalError:
        # table or column does not exist
        return None
    except Exception:
        # propagate unexpected errors to caller
        raise


# -----------------------------------------------------------------------------
# Pretty-print a small table of results
# -----------------------------------------------------------------------------
def _print_results(now_ms: int,
                   base_table: str, base_open_time: Optional[str], base_open_time_ms: Optional[int],
                   dev_table: str, dev_open_time: Optional[str],
                   final_table: str, final_open_time: Optional[str]) -> None:
    # Header with current UTC time
    now_str = utils.ms_to_utc_str(now_ms)
    print(f"[now UTC] ms={now_ms}   ->   {now_str}")
    print("DB tables (all times UTC):")
    # Column headings
    col1 = "table"
    col2 = "max(open_time)"
    col3 = "max(open_time_ms)"
    col4 = "max(open_time_ms) -> UTC"
    print(f"{col1:<30}{col2:<25}{col3:<22}{col4}")
    print("-" * 95)

    # Base row
    if base_open_time_ms is not None:
        try:
            base_ms_to_str = utils.ms_to_utc_str(int(base_open_time_ms))
        except Exception:
            base_ms_to_str = "<invalid ms>"
    else:
        base_ms_to_str = ""
    print(f"{base_table:<30}{str(base_open_time)[:24]:<25}{str(base_open_time_ms):<22}{base_ms_to_str}")

    # Dev row
    print(f"{dev_table:<30}{str(dev_open_time)[:24]:<25}{'':<22}{''}")

    # Final/logreg row
    print(f"{final_table:<30}{str(final_open_time)[:24]:<25}{'':<22}{''}")

    print("-" * 95)


# -----------------------------------------------------------------------------
# Main loop
# -----------------------------------------------------------------------------
def main_loop() -> None:
    # Load configuration for DB and table names
    cfg     = utils._load_config()
    db_path = cfg.get("database", {}).get("db_path")
    if not db_path or not isinstance(db_path, str):
        print("Database path not configured (database.db_path). Exiting.")
        return

    # Resolve table names with project defaults/fallbacks
    base_table  = cfg.get("database", {}).get("dev_data", {}).get("table_name")
    dev_table   = cfg.get("database", {}).get("dev_data", {}).get("table_name_dev")
    final_table = cfg.get("model"   , {}).get("logreg_base_table")

    print("Starting minimal main (UTC+0, epoch-ms). Press Ctrl-C to stop.")
    try:
        while True:
            # Current UTC time in ms (canonical)
            now_ms = utils.now_utc_ms()

            try:
                # Query the DB for maximums
                with sqlite3.connect(db_path) as conn:
                    base_max_time    = _sql_max(conn, base_table , "open_time")
                    base_max_time_ms = _sql_max(conn, base_table , "open_time_ms")
                    dev_max_time     = _sql_max(conn, dev_table  , "open_time")
                    final_max_time   = _sql_max(conn, final_table, "open_time")

                # Print nicely formatted results
                _print_results(now_ms,
                               base_table, base_max_time, base_max_time_ms,
                               dev_table, dev_max_time,
                               final_table, final_max_time)

                # -----------------------------------------------------------------
                # Sync base table: call sync_bchusdt_1m_from_ms with next ms
                # - If base_max_time_ms exists, start from base_max_time_ms + 1
                # - Otherwise start from current UTC ms (minimal, config-driven)
                # -----------------------------------------------------------------
                if base_max_time_ms is not None:
                    start_ms = int(base_max_time_ms) + 1
                else:
                    start_ms = utils.now_utc_ms()

                print(f"Syncing base table '{base_table}' from ms={start_ms} ...")
                sync_bchusdt_1m_from_ms(start_ms)

            except Exception as e:
                # If DB read or sync fails, print a short error line (with UTC time)
                now_str = utils.ms_to_utc_str(now_ms)
                print(f"[{now_str}] Error querying database or syncing: {e}")

            # Wait before next cycle
            time.sleep(POLL_SECONDS)
    except KeyboardInterrupt:
        print("Interrupted by user. Exiting.")


# -----------------------------------------------------------------------------
# Entry point
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    main_loop()

Starting minimal main (UTC+0, epoch-ms). Press Ctrl-C to stop.
Interrupted by user. Exiting.
